# NYC Short-Term Rental Prices - Exploratory Data Analysis

**Author:** Eduardo Claudio Nicacio

**Purpose:** Explore the raw sample of NYC Airbnb listings stored in Weights & Biases in order
to understand its structure and quality, document the problems found, and decide which
cleaning steps are safe to apply before the data enters the training pipeline. The cleaning
decisions taken here are later re-implemented as the reusable `basic_cleaning` MLflow step.

## 1. Fetch the sample data from Weights & Biases

The `download` step of the pipeline uploaded the raw sample as the artifact `sample.csv` in the
`nyc_airbnb` W&B project. We initialize a dedicated run (`group="eda"`) and download the
`latest` version of the artifact. Using `save_code=True` also versions this notebook in W&B.

Note that the artifact is always referenced with an alias (`:latest`); referring to an artifact
without a version or alias is not allowed.

In [ ]:
import wandb
import pandas as pd

run = wandb.init(project="nyc_airbnb", group="eda", save_code=True)
local_path = wandb.use_artifact("sample.csv:latest").file()
df = pd.read_csv(local_path)
df.shape

The sample contains 20,000 listings and 16 columns covering the listing identity, the host,
the location (neighbourhood group / neighbourhood / latitude / longitude), the room type, the
nightly price and several review-related fields.

## 2. Profile the dataset

`ydata-profiling` produces a complete overview of every column: type, distribution, missing
values, cardinality and warnings about potential quality problems.

In [ ]:
import ydata_profiling

profile = ydata_profiling.ProfileReport(df)
profile.to_notebook_iframe()

### Findings

- **Missing values** are present in several columns. `name`, `host_name`, `last_review` and
  `reviews_per_month` are the most affected; `reviews_per_month` is missing exactly for the
  listings that never received a review.
- **`last_review` is stored as text** (`object`/string), even though it represents a date. It
  must be converted to a datetime before any date-based feature engineering.
- **`price` contains outliers**: there are listings priced at `0` and a few extremely high
  prices far outside the normal rental market. After talking to the stakeholders we agreed on
  an acceptable nightly price window of **$10 to $350 inclusive**.
- The geographical columns look consistent with the NYC area, and `neighbourhood_group` only
  contains the five expected boroughs.

## 3. Apply the basic fixes

Two safe fixes are applied here:

1. Drop the listings whose `price` falls outside the agreed window `[10, 350]`.
2. Convert `last_review` from text to a datetime column.

These are exactly the operations that will be moved into the `basic_cleaning` MLflow step, so
that the pipeline performs them automatically on every new data drop.

In [ ]:
min_price = 10
max_price = 350
idx = df['price'].between(min_price, max_price)
df = df[idx].copy()
df['last_review'] = pd.to_datetime(df['last_review'])
df.info()

### Why the missing values are **not** imputed here

Missing values are intentionally left untouched in this EDA. Imputation is handled by the
**inference pipeline** instead (numeric columns are imputed inside the scikit-learn
`ColumnTransformer`, categorical columns use a most-frequent imputer, and the missing review
date is imputed with an old date before the date feature is computed).

Keeping the imputation in the model pipeline means:

- the same treatment is applied at training time and at prediction time, so the deployed
  model can score incomplete records that arrive in production;
- the imputation statistics are learned only from the training data and versioned together
  with the model, which avoids leaking information into the validation/test sets;
- the data-cleaning step stays focused on fixing *invalid* records (price outliers), not on
  guessing values.

## 4. Finish the run

The W&B run is closed so that the notebook and its artifacts are properly versioned.

In [ ]:
run.finish()